# Artifact Keeper: three acts

This notebook runs the same three acts as the live demo, one cell per beat, so you
can walk through it again on your own afterward. Act 1 shows the registry gating
traffic in real time: a typosquat blocked before any upstream fetch, and a known
CVE that auto-quarantines the moment its scan completes. Act 2 turns to what was
already sitting in the cache before any policy existed: an on-demand rescan, a
blast-radius query, and a quarantine you decide to place yourself. Act 3 is about
not being there next time: a brand-new release held behind a 14-day age gate until
a human signs off.

Full narration and troubleshooting live in RUNBOOK.md. Before running any cell
here, the stack needs to be up and configured: `setup/preflight.sh` followed by
`setup/configure.sh` (or, if you already brought the stack up with plain
`docker compose`, just run `setup/configure.sh` once the backend is healthy).


In [ ]:
%%bash
bash setup/warm-cache.sh


## Act 1, beat 1: normal life through the registry

Nothing about the developer's workflow changes. `pip download` and `hf download`
both go through Artifact Keeper, and both come back warm from the cache: the
registry is just where packages come from now, for source dependencies and for
model weights alike.


In [ ]:
%%bash
bash acts/act1-gate.sh 1


## Act 1, beat 2: the typosquat

`pip install requessts` fails, but pip's own error is unhelpful: it just looks like
a miss. Hitting the proxy's simple index directly shows what actually happened: a
curation rule blocked the package by name, before any upstream fetch, and the
registry can tell you exactly why pip couldn't.


In [ ]:
%%bash
bash acts/act1-gate.sh 2


## Act 1, beat 3: the CVE package

Publishing pyyaml 5.3 to the internal team-packages repo triggers a scan, and the
moment that scan completes with a critical finding, the artifact auto-quarantines.
The download that succeeded a moment ago now returns 409. The artifact ends this
cell HELD; open the web UI to see the quarantine banner on it.


In [ ]:
%%bash
bash acts/act1-gate.sh 3


## Act 1, beat 4: resolve it on the record

Releasing the hold by itself is not enough. The scan policy is a second,
independent gate, and it keeps blocking while any critical or high finding stands
unacknowledged. This cell acknowledges every such finding on the record, releases
the quarantine, and shows the download restored to 200.

Run this cell even if you already clicked Release in the web UI: the click clears
only the quarantine hold, while this cell also acknowledges the findings the scan
policy gate requires (the script tolerates the artifact already being released).


In [ ]:
%%bash
bash acts/act1-gate.sh release


## Act 2, beat 1: the cache remembers what nobody scanned

urllib3 1.24.1 came through the pypi-proxy cache before any policy existed, and
nothing has ever evaluated it. An on-demand rescan runs against those cached
bytes, no re-download involved, and turns up a dozen known findings. Reading the
SBOM generated from that same cached artifact shows the proxy cache is no longer a
blind spot. Then this cell flips on scan-aware serving for the proxy, and the same
cached file that downloaded cleanly a moment ago is now blocked for everyone. That
flip is live and stays on afterward; only `setup/reset.sh` turns it back off.


In [ ]:
%%bash
bash acts/act2-lastmonth.sh 1


## Act 2, beat 2: blast radius

This is the Tuesday-morning question: who has this CVE, who actually pulled it,
and who could have based on their access. One query against a known CVE answers
all three for the hosted repos, which is what matters for an incident review.


In [ ]:
%%bash
bash acts/act2-lastmonth.sh 2


## Act 2, beat 3: quarantine-now

Not every hold starts with a scan finding. This cell publishes a clean artifact,
with no findings anywhere, and quarantines it anyway on your own reason, the way
an incident review would. The download's 409 is deliberately generic; it does not
echo the reason back. The reason lives on the quarantine record and in the audit
log instead, which is where it belongs for a review.


In [ ]:
%%bash
bash acts/act2-lastmonth.sh 3


## Act 3, beat 1: the age gate

A release that shipped inside the last 14 days is invisible to pip entirely, and
hitting the file directly returns 451 with a review id attached. Before running the
next cell, approve that review as a human: either in the web UI at `/age-gate`, or
through the API fallback printed in this cell's own output.


In [ ]:
%%bash
bash acts/act3-agegate.sh 1


## Act 3, beat 2: after a human said yes

Once the review is approved, the same request that 451'd a moment ago returns 200.
The approval itself is on the record, permanent for that package and version. Next
time, a brand-new release is two weeks behind the blast before it ever reaches you.


In [ ]:
%%bash
bash acts/act3-agegate.sh 2
